In [1]:
version 18
set more off
adopath ++ "../../ado"
global x "hvzat hvzdvc dd hvze nege hvzac"
use "data/rolling_fit_panel/reg.dta", clear
capture mkdir "tables"
est clear

keep gvkey avyear hvzy1 hvzy2 hvzy3 $x
expand 30
bysort gvkey avyear: gen byte copy=_n
gen byte horizon=ceil(copy/10)
gen int fityear=avyear+horizon+mod(copy-1,10)
keep if inrange(fityear,1969,2012) & inrange(avyear,fityear-9-horizon,fityear-horizon)
gen double y=.
replace y=hvzy1 if horizon==1
replace y=hvzy2 if horizon==2
replace y=hvzy3 if horizon==3
egen byte missct=rowmiss(y $x)
gen byte complete=missct==0
drop missct
bysort fityear horizon: egen long n_cc=total(complete)
sort fityear horizon gvkey avyear

gen byte rankok=0
local nchecked = 0
local nfull = 0
local nrank = 0
local nempty = 0
forvalues t=1969/2012 {
    forvalues h=1/3 {
        local ++nchecked
        quietly count if fityear==`t' & horizon==`h'
        local nrecords = r(N)
        if `nrecords' > 0 {
            quietly summarize n_cc if fityear==`t' & horizon==`h', meanonly
            local ncc = r(mean)
            if `ncc' > 7 {
                regress y $x if fityear==`t' & horizon==`h', notable
                if e(N)>7 & e(rank)==7 {
                    replace rankok=1 if fityear==`t' & horizon==`h'
                    local ++nfull

                    if `t'==1969 | `t'==2012 {
                        scalar hvz_nat_N_`h'_`t' = e(N)
                        scalar hvz_nat_cons_`h'_`t' = _b[_cons]
                        scalar hvz_nat_hvzat_`h'_`t' = _b[hvzat]
                        scalar hvz_nat_hvzdvc_`h'_`t' = _b[hvzdvc]
                        scalar hvz_nat_dd_`h'_`t' = _b[dd]
                        scalar hvz_nat_hvze_`h'_`t' = _b[hvze]
                        scalar hvz_nat_nege_`h'_`t' = _b[nege]
                        scalar hvz_nat_hvzac_`h'_`t' = _b[hvzac]
                    }
                }
                else {
                    local ++nrank
                }
            }
            else {
                local ++nempty
            }
        }
        else {
            local ++nempty
        }
    }
}
display "HVZ native grouped rank coverage: groups checked=" `nchecked' " full rank with residual df=" `nfull' " rank/df failures=" `nrank' " empty groups=" `nempty'
bysort fityear horizon: gen byte cc_tag=(_n==1)
format n_cc %9.0f
list fityear horizon n_cc rankok if cc_tag, noobs sepby(horizon)

forvalues h=1/3 {
    preserve
    keep if horizon==`h'
    bysort fityear: gen byte raw_tag=(_n==1)
    quietly count if raw_tag & !complete
    local rawmiss=r(N)
    display "HVZ horizon " `h' " raw first-row tags on incomplete observations=" `rawmiss'
    drop raw_tag
    keep if complete
    asreg y $x, by(fityear)
    bysort fityear: gen byte tag=(_n==1)
    quietly count if tag & !complete
    local postmiss=r(N)
    display "HVZ horizon " `h' " tagged incomplete results after complete-case restriction=" `postmiss'
    assert `postmiss'==0
    count if tag
    local ntag=r(N)
    assert `ntag'==44
    count if tag & !missing(_Nobs, _R2, _adjR2, _b_cons, _b_hvzat, _b_hvzdvc, _b_dd, _b_hvze, _b_nege, _b_hvzac)
    local nvec=r(N)
    assert `nvec'==44
    list fityear n_cc rankok _Nobs _R2 _adjR2 if tag, noobs
    foreach endpoint in 1969 2012 {
        quietly count if tag & rankok & fityear==`endpoint'
        local endpoint_rank=r(N)
        if `endpoint_rank'==1 {
            quietly summarize _b_cons if tag & fityear==`endpoint', meanonly
            local pcons=r(mean)
            local nvector=r(N)
            assert `nvector'==1
            quietly summarize _b_hvzat if tag & fityear==`endpoint', meanonly
            local p_hvzat=r(mean)
            quietly summarize _b_hvzdvc if tag & fityear==`endpoint', meanonly
            local p_hvzdvc=r(mean)
            quietly summarize _b_dd if tag & fityear==`endpoint', meanonly
            local p_dd=r(mean)
            quietly summarize _b_hvze if tag & fityear==`endpoint', meanonly
            local p_hvze=r(mean)
            quietly summarize _b_nege if tag & fityear==`endpoint', meanonly
            local p_nege=r(mean)
            quietly summarize _b_hvzac if tag & fityear==`endpoint', meanonly
            local p_hvzac=r(mean)
            quietly summarize _Nobs if tag & fityear==`endpoint', meanonly
            local asregN=r(mean)
            local natN=scalar(hvz_nat_N_`h'_`endpoint')
            local bdiff=max(abs(scalar(hvz_nat_cons_`h'_`endpoint')-`pcons'), abs(scalar(hvz_nat_hvzat_`h'_`endpoint')-`p_hvzat'), abs(scalar(hvz_nat_hvzdvc_`h'_`endpoint')-`p_hvzdvc'), abs(scalar(hvz_nat_dd_`h'_`endpoint')-`p_dd'), abs(scalar(hvz_nat_hvze_`h'_`endpoint')-`p_hvze'), abs(scalar(hvz_nat_nege_`h'_`endpoint')-`p_nege'), abs(scalar(hvz_nat_hvzac_`h'_`endpoint')-`p_hvzac'))
            local ndiff=abs(`natN'-`asregN')
            display "HVZ horizon " `h' " year " `endpoint' " native/asreg max coefficient difference=" %12.8g `bdiff' " N difference=" %9.0f `ndiff'
        }
        else {
            display "HVZ horizon " `h' " year " `endpoint' " native comparison skipped: full-rank check failed"
        }
    }
    count if tag & rankok
    local nfullvec=r(N)
    display "HVZ horizon " `h' " native full-rank years=" `nfullvec' " of 44 vectors"
    keep if tag & rankok
    count
    local nfits=r(N)
    assert `nfits'==44
    display "HVZ qualifying complete-case asreg fits at horizon " `h' "=" `nfits'
    quietly summarize _R2, meanonly
    local meanr2=100*r(mean)
    local r2fits=r(N)
    assert `r2fits'==44
    quietly summarize _adjR2, meanonly
    local meanar2=100*r(mean)
    local ar2fits=r(N)
    assert `ar2fits'==44
    quietly summarize _Nobs, meanonly
    local meanrows=r(mean)
    quietly summarize _Nobs
    local minrows=r(min)
    local maxrows=r(max)
    display "HVZ horizon " `h' " mean annual N=" %9.1f `meanrows' " min=" %9.0f `minrows' " max=" %9.0f `maxrows' " mean R2/AdjR2 percentages from 44 groups"
    mean _b_cons _b_hvzat _b_hvzdvc _b_dd _b_hvze _b_nege _b_hvzac
    estadd scalar r2=`meanr2', replace
    estadd scalar r2_a=`meanar2', replace
    est store h`h'
    restore
}
esttab h1 h2 h3 using "tables/hvz_forecast.md", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label mtitles("t+1" "t+2" "t+3") coeflabels(_b_cons "Intercept" _b_hvzat "A" _b_hvzdvc "D" _b_dd "DD" _b_hvze "E" _b_nege "NegE" _b_hvzac "AC") addnotes("Each column averages annual OLS fit vectors that pass the native full-rank check; coverage counts and exceptions are documented in the log." "Parentheses show two-sided p-values for native tests of the mean coefficient; see logs for t statistics." "N counts annual fits; underlying regression N is summarized by year and horizon in the log." "R-squared statistics are percentages, as in Table 1.")
esttab h1 h2 h3 using "tables/hvz_forecast.rtf", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label mtitles("t+1" "t+2" "t+3") coeflabels(_b_cons "Intercept" _b_hvzat "A" _b_hvzdvc "D" _b_dd "DD" _b_hvze "E" _b_nege "NegE" _b_hvzac "AC") addnotes("Each column averages annual OLS fit vectors that pass the native full-rank check; coverage counts and exceptions are documented in the log." "Parentheses show two-sided p-values for native tests of the mean coefficient; see logs for t statistics." "N counts annual fits; underlying regression N is summarized by year and horizon in the log." "R-squared statistics are percentages, as in Table 1.")
esttab h1 h2 h3 using "tables/hvz_forecast.csv", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label mtitles("t+1" "t+2" "t+3") coeflabels(_b_cons "Intercept" _b_hvzat "A" _b_hvzdvc "D" _b_dd "DD" _b_hvze "E" _b_nege "NegE" _b_hvzac "AC") addnotes("Each column averages annual OLS fit vectors that pass the native full-rank check; coverage counts and exceptions are documented in the log." "Parentheses show two-sided p-values for native tests of the mean coefficient; see logs for t statistics." "N counts annual fits; underlying regression N is summarized by year and horizon in the log." "R-squared statistics are percentages, as in Table 1.")
esttab h1 h2 h3 using "tables/hvz_forecast.tex", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label mtitles("t+1" "t+2" "t+3") coeflabels(_b_cons "Intercept" _b_hvzat "A" _b_hvzdvc "D" _b_dd "DD" _b_hvze "E" _b_nege "NegE" _b_hvzac "AC") addnotes("Each column averages annual OLS fit vectors that pass the native full-rank check; coverage counts and exceptions are documented in the log." "Parentheses show two-sided p-values for native tests of the mean coefficient; see logs for t statistics." "N counts annual fits; underlying regression N is summarized by year and horizon in the log." "R-squared statistics are percentages, as in Table 1.")
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do

In [1]:
version 18
set more off
adopath ++ "../../ado"
global x "nege epx epnegx"
use "data/rolling_fit_panel/reg.dta", clear
capture mkdir "tables"
est clear

keep gvkey avyear eppy1 eppy2 eppy3 $x
expand 30
bysort gvkey avyear: gen byte copy=_n
gen byte horizon=ceil(copy/10)
gen int fityear=avyear+horizon+mod(copy-1,10)
keep if inrange(fityear,1969,2012) & inrange(avyear,fityear-9-horizon,fityear-horizon)
gen double y=.
replace y=eppy1 if horizon==1
replace y=eppy2 if horizon==2
replace y=eppy3 if horizon==3
egen byte missct=rowmiss(y $x)
gen byte complete=missct==0
drop missct
bysort fityear horizon: egen long n_cc=total(complete)
sort fityear horizon gvkey avyear

gen byte rankok=0
local nchecked = 0
local nfull = 0
local nrank = 0
local nempty = 0
forvalues t=1969/2012 {
    forvalues h=1/3 {
        local ++nchecked
        quietly count if fityear==`t' & horizon==`h'
        local nrecords = r(N)
        if `nrecords' > 0 {
            quietly summarize n_cc if fityear==`t' & horizon==`h', meanonly
            local ncc = r(mean)
            if `ncc' > 4 {
                regress y $x if fityear==`t' & horizon==`h', notable
                if e(N)>4 & e(rank)==4 {
                    replace rankok=1 if fityear==`t' & horizon==`h'
                    local ++nfull

                    if `t'==1969 | `t'==2012 {
                        scalar ep_nat_N_`h'_`t' = e(N)
                        scalar ep_nat_cons_`h'_`t' = _b[_cons]
                        scalar ep_nat_nege_`h'_`t' = _b[nege]
                        scalar ep_nat_epx_`h'_`t' = _b[epx]
                        scalar ep_nat_epnegx_`h'_`t' = _b[epnegx]
                    }
                }
                else {
                    local ++nrank
                }
            }
            else {
                local ++nempty
            }
        }
        else {
            local ++nempty
        }
    }
}
display "EP native grouped rank coverage: groups checked=" `nchecked' " full rank with residual df=" `nfull' " rank/df failures=" `nrank' " empty groups=" `nempty'
bysort fityear horizon: gen byte cc_tag=(_n==1)
format n_cc %9.0f
list fityear horizon n_cc rankok if cc_tag, noobs sepby(horizon)

forvalues h=1/3 {
    preserve
    keep if horizon==`h'
    bysort fityear: gen byte raw_tag=(_n==1)
    quietly count if raw_tag & !complete
    local rawmiss=r(N)
    display "EP horizon " `h' " raw first-row tags on incomplete observations=" `rawmiss'
    drop raw_tag
    keep if complete
    asreg y $x, by(fityear)
    bysort fityear: gen byte tag=(_n==1)
    quietly count if tag & !complete
    local postmiss=r(N)
    display "EP horizon " `h' " tagged incomplete results after complete-case restriction=" `postmiss'
    assert `postmiss'==0
    count if tag
    local ntag=r(N)
    assert `ntag'==44
    count if tag & !missing(_Nobs, _R2, _adjR2, _b_cons, _b_nege, _b_epx, _b_epnegx)
    local nvec=r(N)
    assert `nvec'==44
    list fityear n_cc rankok _Nobs _R2 _adjR2 if tag, noobs
    foreach endpoint in 1969 2012 {
        quietly count if tag & rankok & fityear==`endpoint'
        local endpoint_rank=r(N)
        if `endpoint_rank'==1 {
            quietly summarize _b_cons if tag & fityear==`endpoint', meanonly
            local pcons=r(mean)
            local nvector=r(N)
            assert `nvector'==1
            quietly summarize _b_nege if tag & fityear==`endpoint', meanonly
            local p_nege=r(mean)
            quietly summarize _b_epx if tag & fityear==`endpoint', meanonly
            local p_epx=r(mean)
            quietly summarize _b_epnegx if tag & fityear==`endpoint', meanonly
            local p_epnegx=r(mean)
            quietly summarize _Nobs if tag & fityear==`endpoint', meanonly
            local asregN=r(mean)
            local natN=scalar(ep_nat_N_`h'_`endpoint')
            local bdiff=max(abs(scalar(ep_nat_cons_`h'_`endpoint')-`pcons'), abs(scalar(ep_nat_nege_`h'_`endpoint')-`p_nege'), abs(scalar(ep_nat_epx_`h'_`endpoint')-`p_epx'), abs(scalar(ep_nat_epnegx_`h'_`endpoint')-`p_epnegx'))
            local ndiff=abs(`natN'-`asregN')
            display "EP horizon " `h' " year " `endpoint' " native/asreg max coefficient difference=" %12.8g `bdiff' " N difference=" %9.0f `ndiff'
        }
        else {
            display "EP horizon " `h' " year " `endpoint' " native comparison skipped: full-rank check failed"
        }
    }
    count if tag & rankok
    local nfullvec=r(N)
    display "EP horizon " `h' " native full-rank years=" `nfullvec' " of 44 vectors"
    keep if tag & rankok
    count
    local nfits=r(N)
    assert `nfits'==44
    display "EP qualifying complete-case asreg fits at horizon " `h' "=" `nfits'
    quietly summarize _R2, meanonly
    local meanr2=100*r(mean)
    local r2fits=r(N)
    assert `r2fits'==44
    quietly summarize _adjR2, meanonly
    local meanar2=100*r(mean)
    local ar2fits=r(N)
    assert `ar2fits'==44
    quietly summarize _Nobs, meanonly
    local meanrows=r(mean)
    quietly summarize _Nobs
    local minrows=r(min)
    local maxrows=r(max)
    display "EP horizon " `h' " mean annual N=" %9.1f `meanrows' " min=" %9.0f `minrows' " max=" %9.0f `maxrows' " mean R2/AdjR2 percentages from 44 groups"
    mean _b_cons _b_nege _b_epx _b_epnegx
    estadd scalar r2=`meanr2', replace
    estadd scalar r2_a=`meanar2', replace
    est store h`h'
    restore
}
esttab h1 h2 h3 using "tables/ep_forecast.md", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label mtitles("t+1" "t+2" "t+3") coeflabels(_b_cons "Intercept" _b_nege "NegE" _b_epx "E" _b_epnegx "NegE*E") addnotes("Each column averages annual OLS fit vectors that pass the native full-rank check; coverage counts and exceptions are documented in the log." "Parentheses show two-sided p-values for native tests of the mean coefficient; see logs for t statistics." "N counts annual fits; underlying regression N is summarized by year and horizon in the log." "R-squared statistics are percentages, as in Table 1.")
esttab h1 h2 h3 using "tables/ep_forecast.rtf", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label mtitles("t+1" "t+2" "t+3") coeflabels(_b_cons "Intercept" _b_nege "NegE" _b_epx "E" _b_epnegx "NegE*E") addnotes("Each column averages annual OLS fit vectors that pass the native full-rank check; coverage counts and exceptions are documented in the log." "Parentheses show two-sided p-values for native tests of the mean coefficient; see logs for t statistics." "N counts annual fits; underlying regression N is summarized by year and horizon in the log." "R-squared statistics are percentages, as in Table 1.")
esttab h1 h2 h3 using "tables/ep_forecast.csv", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label mtitles("t+1" "t+2" "t+3") coeflabels(_b_cons "Intercept" _b_nege "NegE" _b_epx "E" _b_epnegx "NegE*E") addnotes("Each column averages annual OLS fit vectors that pass the native full-rank check; coverage counts and exceptions are documented in the log." "Parentheses show two-sided p-values for native tests of the mean coefficient; see logs for t statistics." "N counts annual fits; underlying regression N is summarized by year and horizon in the log." "R-squared statistics are percentages, as in Table 1.")
esttab h1 h2 h3 using "tables/ep_forecast.tex", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label mtitles("t+1" "t+2" "t+3") coeflabels(_b_cons "Intercept" _b_nege "NegE" _b_epx "E" _b_epnegx "NegE*E") addnotes("Each column averages annual OLS fit vectors that pass the native full-rank check; coverage counts and exceptions are documented in the log." "Parentheses show two-sided p-values for native tests of the mean coefficient; see logs for t statistics." "N counts annual fits; underlying regression N is summarized by year and horizon in the log." "R-squared statistics are percentages, as in Table 1.")
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do

In [1]:
version 18
set more off
adopath ++ "../../ado"
global x "nege rix rinegx rib ritacc"
use "data/rolling_fit_panel/reg.dta", clear
capture mkdir "tables"
est clear

keep gvkey avyear riy1 riy2 riy3 $x
expand 30
bysort gvkey avyear: gen byte copy=_n
gen byte horizon=ceil(copy/10)
gen int fityear=avyear+horizon+mod(copy-1,10)
keep if inrange(fityear,1969,2012) & inrange(avyear,fityear-9-horizon,fityear-horizon)
gen double y=.
replace y=riy1 if horizon==1
replace y=riy2 if horizon==2
replace y=riy3 if horizon==3
egen byte missct=rowmiss(y $x)
gen byte complete=missct==0
drop missct
bysort fityear horizon: egen long n_cc=total(complete)
sort fityear horizon gvkey avyear

gen byte rankok=0
local nchecked = 0
local nfull = 0
local nrank = 0
local nempty = 0
forvalues t=1969/2012 {
    forvalues h=1/3 {
        local ++nchecked
        quietly count if fityear==`t' & horizon==`h'
        local nrecords = r(N)
        if `nrecords' > 0 {
            quietly summarize n_cc if fityear==`t' & horizon==`h', meanonly
            local ncc = r(mean)
            if `ncc' > 6 {
                regress y $x if fityear==`t' & horizon==`h', notable
                if e(N)>6 & e(rank)==6 {
                    replace rankok=1 if fityear==`t' & horizon==`h'
                    local ++nfull

                    if `t'==1969 | `t'==2012 {
                        scalar ri_nat_N_`h'_`t' = e(N)
                        scalar ri_nat_cons_`h'_`t' = _b[_cons]
                        scalar ri_nat_nege_`h'_`t' = _b[nege]
                        scalar ri_nat_rix_`h'_`t' = _b[rix]
                        scalar ri_nat_rinegx_`h'_`t' = _b[rinegx]
                        scalar ri_nat_rib_`h'_`t' = _b[rib]
                        scalar ri_nat_ritacc_`h'_`t' = _b[ritacc]
                    }
                }
                else {
                    local ++nrank
                }
            }
            else {
                local ++nempty
            }
        }
        else {
            local ++nempty
        }
    }
}
display "RI native grouped rank coverage: groups checked=" `nchecked' " full rank with residual df=" `nfull' " rank/df failures=" `nrank' " empty groups=" `nempty'
bysort fityear horizon: gen byte cc_tag=(_n==1)
format n_cc %9.0f
list fityear horizon n_cc rankok if cc_tag, noobs sepby(horizon)

forvalues h=1/3 {
    preserve
    keep if horizon==`h'
    bysort fityear: gen byte raw_tag=(_n==1)
    quietly count if raw_tag & !complete
    local rawmiss=r(N)
    display "RI horizon " `h' " raw first-row tags on incomplete observations=" `rawmiss'
    drop raw_tag
    keep if complete
    asreg y $x, by(fityear)
    bysort fityear: gen byte tag=(_n==1)
    quietly count if tag & !complete
    local postmiss=r(N)
    display "RI horizon " `h' " tagged incomplete results after complete-case restriction=" `postmiss'
    assert `postmiss'==0
    count if tag
    local ntag=r(N)
    assert `ntag'==44
    count if tag & !missing(_Nobs, _R2, _adjR2, _b_cons, _b_nege, _b_rix, _b_rinegx, _b_rib, _b_ritacc)
    local nvec=r(N)
    assert `nvec'==44
    list fityear n_cc rankok _Nobs _R2 _adjR2 if tag, noobs
    foreach endpoint in 1969 2012 {
        quietly count if tag & rankok & fityear==`endpoint'
        local endpoint_rank=r(N)
        if `endpoint_rank'==1 {
            quietly summarize _b_cons if tag & fityear==`endpoint', meanonly
            local pcons=r(mean)
            local nvector=r(N)
            assert `nvector'==1
            quietly summarize _b_nege if tag & fityear==`endpoint', meanonly
            local p_nege=r(mean)
            quietly summarize _b_rix if tag & fityear==`endpoint', meanonly
            local p_rix=r(mean)
            quietly summarize _b_rinegx if tag & fityear==`endpoint', meanonly
            local p_rinegx=r(mean)
            quietly summarize _b_rib if tag & fityear==`endpoint', meanonly
            local p_rib=r(mean)
            quietly summarize _b_ritacc if tag & fityear==`endpoint', meanonly
            local p_ritacc=r(mean)
            quietly summarize _Nobs if tag & fityear==`endpoint', meanonly
            local asregN=r(mean)
            local natN=scalar(ri_nat_N_`h'_`endpoint')
            local bdiff=max(abs(scalar(ri_nat_cons_`h'_`endpoint')-`pcons'), abs(scalar(ri_nat_nege_`h'_`endpoint')-`p_nege'), abs(scalar(ri_nat_rix_`h'_`endpoint')-`p_rix'), abs(scalar(ri_nat_rinegx_`h'_`endpoint')-`p_rinegx'), abs(scalar(ri_nat_rib_`h'_`endpoint')-`p_rib'), abs(scalar(ri_nat_ritacc_`h'_`endpoint')-`p_ritacc'))
            local ndiff=abs(`natN'-`asregN')
            display "RI horizon " `h' " year " `endpoint' " native/asreg max coefficient difference=" %12.8g `bdiff' " N difference=" %9.0f `ndiff'
        }
        else {
            display "RI horizon " `h' " year " `endpoint' " native comparison skipped: full-rank check failed"
        }
    }
    count if tag & rankok
    local nfullvec=r(N)
    display "RI horizon " `h' " native full-rank years=" `nfullvec' " of 44 vectors"
    keep if tag & rankok
    count
    local nfits=r(N)
    if `h'==3 {
        assert `nfits'==43
    }
    else {
        assert `nfits'==44
    }
    display "RI qualifying complete-case asreg fits at horizon " `h' "=" `nfits'
    quietly summarize _R2, meanonly
    local meanr2=100*r(mean)
    local r2fits=r(N)
    if `h'==3 {
        assert `r2fits'==43
    }
    else {
        assert `r2fits'==44
    }
    quietly summarize _adjR2, meanonly
    local meanar2=100*r(mean)
    local ar2fits=r(N)
    if `h'==3 {
        assert `ar2fits'==43
    }
    else {
        assert `ar2fits'==44
    }
    quietly summarize _Nobs, meanonly
    local meanrows=r(mean)
    quietly summarize _Nobs
    local minrows=r(min)
    local maxrows=r(max)
    display "RI horizon " `h' " mean annual N=" %9.1f `meanrows' " min=" %9.0f `minrows' " max=" %9.0f `maxrows' " mean R2/AdjR2 percentages from " `nfits' " groups"
    mean _b_cons _b_nege _b_rix _b_rinegx _b_rib _b_ritacc
    estadd scalar r2=`meanr2', replace
    estadd scalar r2_a=`meanar2', replace
    est store h`h'
    restore
}
esttab h1 h2 h3 using "tables/ri_forecast.md", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label mtitles("t+1" "t+2" "t+3") coeflabels(_b_cons "Intercept" _b_nege "NegE" _b_rix "E" _b_rinegx "NegE*E" _b_rib "B" _b_ritacc "TACC") addnotes("Each column averages annual OLS fit vectors that pass the native full-rank check; coverage counts and exceptions are documented in the log." "Parentheses show two-sided p-values for native tests of the mean coefficient; see logs for t statistics." "N counts annual fits; underlying regression N is summarized by year and horizon in the log." "R-squared statistics are percentages, as in Table 1.")
esttab h1 h2 h3 using "tables/ri_forecast.rtf", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label mtitles("t+1" "t+2" "t+3") coeflabels(_b_cons "Intercept" _b_nege "NegE" _b_rix "E" _b_rinegx "NegE*E" _b_rib "B" _b_ritacc "TACC") addnotes("Each column averages annual OLS fit vectors that pass the native full-rank check; coverage counts and exceptions are documented in the log." "Parentheses show two-sided p-values for native tests of the mean coefficient; see logs for t statistics." "N counts annual fits; underlying regression N is summarized by year and horizon in the log." "R-squared statistics are percentages, as in Table 1.")
esttab h1 h2 h3 using "tables/ri_forecast.csv", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label mtitles("t+1" "t+2" "t+3") coeflabels(_b_cons "Intercept" _b_nege "NegE" _b_rix "E" _b_rinegx "NegE*E" _b_rib "B" _b_ritacc "TACC") addnotes("Each column averages annual OLS fit vectors that pass the native full-rank check; coverage counts and exceptions are documented in the log." "Parentheses show two-sided p-values for native tests of the mean coefficient; see logs for t statistics." "N counts annual fits; underlying regression N is summarized by year and horizon in the log." "R-squared statistics are percentages, as in Table 1.")
esttab h1 h2 h3 using "tables/ri_forecast.tex", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label mtitles("t+1" "t+2" "t+3") coeflabels(_b_cons "Intercept" _b_nege "NegE" _b_rix "E" _b_rinegx "NegE*E" _b_rib "B" _b_ritacc "TACC") addnotes("Each column averages annual OLS fit vectors that pass the native full-rank check; coverage counts and exceptions are documented in the log." "Parentheses show two-sided p-values for native tests of the mean coefficient; see logs for t statistics." "N counts annual fits; underlying regression N is summarized by year and horizon in the log." "R-squared statistics are percentages, as in Table 1.")
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do

In [1]:
version 18
set more off
adopath ++ "../../ado"
use "data/rolling_fit_panel/reg.dta", clear
capture mkdir "tables"
est clear
isid gvkey avyear

global x "hvzat hvzdvc dd hvze nege hvzac"
keep gvkey avyear hvzy1 hvzy2 hvzy3 $x
label var hvzat "A: total assets (USD millions)"
label var hvzdvc "D: dividends (USD millions)"
label var dd "DD: positive dividend indicator"
label var hvze "E: current earnings (USD millions)"
label var nege "NegE: negative earnings indicator"
label var hvzac "AC: accounting changes (USD millions)"
forvalues h=1/3 {
    preserve
    local firstfit=1969
    local winlo=`firstfit'-9-`h'
    local winhi=2012-`h'
    gen double y=hvzy`h'
    label var y "Future earnings, h=`h' (USD millions)"
    egen byte missct=rowmiss(y $x)
    gen byte inunion=0
    forvalues fit=`firstfit'/2012 {
        replace inunion=1 if inrange(avyear,`fit'-9-`h',`fit'-`h')
    }
    keep if inunion & missct==0
    isid gvkey avyear
    count
    local nunion=r(N)
    display "HVZ horizon " `h' " complete-case window union: base AVYEAR " `winlo' "-" `winhi' "; unique rows=" `nunion'
    estpost tabstat y $x, statistics(n mean sd p1 p99) columns(statistics)
    estadd scalar union_rows=`nunion', replace
    est store desc_h`h'
    restore
}
esttab desc_h1 desc_h2 desc_h3 using "tables/hvz_input_union.md", cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p1(fmt(3)) p99(fmt(3))") nostar stats(union_rows, fmt(0) labels("Unique rows in window union")) replace label mtitles("h=1" "h=2" "h=3") addnotes("Sample is unique GVKEY-by-availability-year input rows appearing in at least one qualifying annual training window for the model and horizon." "Rows are complete for the model outcome and regressors; N counts unique inputs, not expanded/repeated fit rows or the annual-fit coefficient summary." "Input statistics reflect the annual 1/99 winsorized construction; union endpoints and counts are logged.")
esttab desc_h1 desc_h2 desc_h3 using "tables/hvz_input_union.rtf", cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p1(fmt(3)) p99(fmt(3))") nostar stats(union_rows, fmt(0) labels("Unique rows in window union")) replace label mtitles("h=1" "h=2" "h=3") addnotes("Sample is unique GVKEY-by-availability-year input rows appearing in at least one qualifying annual training window for the model and horizon." "Rows are complete for the model outcome and regressors; N counts unique inputs, not expanded/repeated fit rows or the annual-fit coefficient summary." "Input statistics reflect the annual 1/99 winsorized construction; union endpoints and counts are logged.")
esttab desc_h1 desc_h2 desc_h3 using "tables/hvz_input_union.csv", cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p1(fmt(3)) p99(fmt(3))") nostar stats(union_rows, fmt(0) labels("Unique rows in window union")) replace label mtitles("h=1" "h=2" "h=3") addnotes("Sample is unique GVKEY-by-availability-year input rows appearing in at least one qualifying annual training window for the model and horizon." "Rows are complete for the model outcome and regressors; N counts unique inputs, not expanded/repeated fit rows or the annual-fit coefficient summary." "Input statistics reflect the annual 1/99 winsorized construction; union endpoints and counts are logged.")
esttab desc_h1 desc_h2 desc_h3 using "tables/hvz_input_union.tex", cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p1(fmt(3)) p99(fmt(3))") nostar stats(union_rows, fmt(0) labels("Unique rows in window union")) replace label mtitles("h=1" "h=2" "h=3") addnotes("Sample is unique GVKEY-by-availability-year input rows appearing in at least one qualifying annual training window for the model and horizon." "Rows are complete for the model outcome and regressors; N counts unique inputs, not expanded/repeated fit rows or the annual-fit coefficient summary." "Input statistics reflect the annual 1/99 winsorized construction; union endpoints and counts are logged.")
est clear

use "data/rolling_fit_panel/reg.dta", clear
isid gvkey avyear
global x "nege epx epnegx"
keep gvkey avyear eppy1 eppy2 eppy3 $x
label var nege "NegE: negative earnings indicator"
label var epx "E: current earnings per share (USD)"
label var epnegx "NegE x E: interaction"
forvalues h=1/3 {
    preserve
    local firstfit=1969
    local winlo=`firstfit'-9-`h'
    local winhi=2012-`h'
    gen double y=eppy`h'
    label var y "Future earnings per share, h=`h' (USD)"
    egen byte missct=rowmiss(y $x)
    gen byte inunion=0
    forvalues fit=`firstfit'/2012 {
        replace inunion=1 if inrange(avyear,`fit'-9-`h',`fit'-`h')
    }
    keep if inunion & missct==0
    isid gvkey avyear
    count
    local nunion=r(N)
    display "EP horizon " `h' " complete-case window union: base AVYEAR " `winlo' "-" `winhi' "; unique rows=" `nunion'
    estpost tabstat y $x, statistics(n mean sd p1 p99) columns(statistics)
    estadd scalar union_rows=`nunion', replace
    est store desc_h`h'
    restore
}
esttab desc_h1 desc_h2 desc_h3 using "tables/ep_input_union.md", cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p1(fmt(3)) p99(fmt(3))") nostar stats(union_rows, fmt(0) labels("Unique rows in window union")) replace label mtitles("h=1" "h=2" "h=3") addnotes("Sample is unique GVKEY-by-availability-year input rows appearing in at least one qualifying annual training window for the model and horizon." "Rows are complete for the model outcome and regressors; N counts unique inputs, not expanded/repeated fit rows or the annual-fit coefficient summary." "Input statistics reflect the annual 1/99 winsorized construction; union endpoints and counts are logged.")
esttab desc_h1 desc_h2 desc_h3 using "tables/ep_input_union.rtf", cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p1(fmt(3)) p99(fmt(3))") nostar stats(union_rows, fmt(0) labels("Unique rows in window union")) replace label mtitles("h=1" "h=2" "h=3") addnotes("Sample is unique GVKEY-by-availability-year input rows appearing in at least one qualifying annual training window for the model and horizon." "Rows are complete for the model outcome and regressors; N counts unique inputs, not expanded/repeated fit rows or the annual-fit coefficient summary." "Input statistics reflect the annual 1/99 winsorized construction; union endpoints and counts are logged.")
esttab desc_h1 desc_h2 desc_h3 using "tables/ep_input_union.csv", cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p1(fmt(3)) p99(fmt(3))") nostar stats(union_rows, fmt(0) labels("Unique rows in window union")) replace label mtitles("h=1" "h=2" "h=3") addnotes("Sample is unique GVKEY-by-availability-year input rows appearing in at least one qualifying annual training window for the model and horizon." "Rows are complete for the model outcome and regressors; N counts unique inputs, not expanded/repeated fit rows or the annual-fit coefficient summary." "Input statistics reflect the annual 1/99 winsorized construction; union endpoints and counts are logged.")
esttab desc_h1 desc_h2 desc_h3 using "tables/ep_input_union.tex", cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p1(fmt(3)) p99(fmt(3))") nostar stats(union_rows, fmt(0) labels("Unique rows in window union")) replace label mtitles("h=1" "h=2" "h=3") addnotes("Sample is unique GVKEY-by-availability-year input rows appearing in at least one qualifying annual training window for the model and horizon." "Rows are complete for the model outcome and regressors; N counts unique inputs, not expanded/repeated fit rows or the annual-fit coefficient summary." "Input statistics reflect the annual 1/99 winsorized construction; union endpoints and counts are logged.")
est clear

use "data/rolling_fit_panel/reg.dta", clear
isid gvkey avyear
global x "nege rix rinegx rib ritacc"
keep gvkey avyear riy1 riy2 riy3 $x
label var nege "NegE: negative earnings indicator"
label var rix "E: current earnings per share (USD)"
label var rinegx "NegE x E: interaction"
label var rib "B: book value per share (USD)"
label var ritacc "TACC: accruals per share (USD)"
forvalues h=1/3 {
    preserve
    local firstfit=1969
    if `h'==3 local firstfit=1970
    local winlo=`firstfit'-9-`h'
    local winhi=2012-`h'
    gen double y=riy`h'
    label var y "Future earnings per share, h=`h' (USD)"
    egen byte missct=rowmiss(y $x)
    gen byte inunion=0
    forvalues fit=`firstfit'/2012 {
        replace inunion=1 if inrange(avyear,`fit'-9-`h',`fit'-`h')
    }
    keep if inunion & missct==0
    isid gvkey avyear
    count
    local nunion=r(N)
    display "RI horizon " `h' " complete-case full-rank window union: base AVYEAR " `winlo' "-" `winhi' "; unique rows=" `nunion'
    estpost tabstat y $x, statistics(n mean sd p1 p99) columns(statistics)
    estadd scalar union_rows=`nunion', replace
    est store desc_h`h'
    restore
}
esttab desc_h1 desc_h2 desc_h3 using "tables/ri_input_union.md", cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p1(fmt(3)) p99(fmt(3))") nostar stats(union_rows, fmt(0) labels("Unique rows in window union")) replace label mtitles("h=1" "h=2" "h=3") addnotes("Sample is unique GVKEY-by-availability-year input rows appearing in at least one native full-rank annual training window for the model and horizon." "Rows are complete for the model outcome and regressors; N counts unique inputs, not expanded/repeated fit rows or the annual-fit coefficient summary." "RI h=3 begins at T=1970 because native full-rank checking rejects T=1969; base-year union endpoints and counts are logged. Statistics reflect annual 1/99 winsorization.")
esttab desc_h1 desc_h2 desc_h3 using "tables/ri_input_union.rtf", cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p1(fmt(3)) p99(fmt(3))") nostar stats(union_rows, fmt(0) labels("Unique rows in window union")) replace label mtitles("h=1" "h=2" "h=3") addnotes("Sample is unique GVKEY-by-availability-year input rows appearing in at least one native full-rank annual training window for the model and horizon." "Rows are complete for the model outcome and regressors; N counts unique inputs, not expanded/repeated fit rows or the annual-fit coefficient summary." "RI h=3 begins at T=1970 because native full-rank checking rejects T=1969; base-year union endpoints and counts are logged. Statistics reflect annual 1/99 winsorization.")
esttab desc_h1 desc_h2 desc_h3 using "tables/ri_input_union.csv", cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p1(fmt(3)) p99(fmt(3))") nostar stats(union_rows, fmt(0) labels("Unique rows in window union")) replace label mtitles("h=1" "h=2" "h=3") addnotes("Sample is unique GVKEY-by-availability-year input rows appearing in at least one native full-rank annual training window for the model and horizon." "Rows are complete for the model outcome and regressors; N counts unique inputs, not expanded/repeated fit rows or the annual-fit coefficient summary." "RI h=3 begins at T=1970 because native full-rank checking rejects T=1969; base-year union endpoints and counts are logged. Statistics reflect annual 1/99 winsorization.")
esttab desc_h1 desc_h2 desc_h3 using "tables/ri_input_union.tex", cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p1(fmt(3)) p99(fmt(3))") nostar stats(union_rows, fmt(0) labels("Unique rows in window union")) replace label mtitles("h=1" "h=2" "h=3") addnotes("Sample is unique GVKEY-by-availability-year input rows appearing in at least one native full-rank annual training window for the model and horizon." "Rows are complete for the model outcome and regressors; N counts unique inputs, not expanded/repeated fit rows or the annual-fit coefficient summary." "RI h=3 begins at T=1970 because native full-rank checking rejects T=1969; base-year union endpoints and counts are logged. Statistics reflect annual 1/99 winsorization.")
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do